### 문제
1. 일반행정 데이터와 대중교통 데이터를 로드
2. 두개의 데이터프레임을 결합(단순한 행 결합)
3. 데이터의 필터링 고객질문에 대한 상담사의 답변이 즉각적으로 오는 데이터들만 필터
4. 질문 중 중복 데이터를 제거
5. 질문들을 모아서 토큰화, 벡터화
5. 그 외의 질문 목륵을 이용하여 코사인 유사도 확인하고 유사 질문과 답변을 출력

In [44]:
import pandas as pd

In [45]:
# 2개의 데이터프레임을 로드
df1 = pd.read_json('../data/민원(콜센터) 질의응답_다산콜센터_일반행정 문의_Training.json')
df2 = pd.read_json("../data/민원(콜센터) 질의응답_다산콜센터_대중교통 안내_Training.json")

In [46]:
# 2개의 데이터프레임을 단순한 행 결합 (union 결합)
total_df = pd.concat([df2, df1], ignore_index=True)

In [47]:
# 단순 결합시 주의할 점 : 인덱스의 값이 중복 값이 생기는 부분
total_df.loc[0, ]

도메인                                     다산콜센터
카테고리                                  대중교통 안내
대화셋일련번호                                 B2033
화자                                         고객
문장번호                                        1
고객의도                                     버스노선
상담사의도                                        
QA                                          Q
고객질문(요청)        서울 가산동에서 남대문시장가는 버스노선을 알고싶습니다
상담사질문(요청)                                    
고객답변                                         
상담사답변                                        
개체명                    서울, 가산동, 남대문시장, 버스, 노선
용어사전         서울/지명/ 가산동/동네/ 남대문시장/지명/ 버스/교통수단
지식베이스                                가산동,교통수단
Name: 0, dtype: object

In [48]:
total_df.head()

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,대중교통 안내,B2033,고객,1,버스노선,,Q,서울 가산동에서 남대문시장가는 버스노선을 알고싶습니다,,,,"서울, 가산동, 남대문시장, 버스, 노선",서울/지명/ 가산동/동네/ 남대문시장/지명/ 버스/교통수단,"가산동,교통수단"
1,다산콜센터,대중교통 안내,B2033,상담사,2,,버스노선,Q,,가산동 어디에서 출발하십니까?,,,"가산동, 출발",가산동/동네/ 출발/출발지,"출발,출발지"
2,다산콜센터,대중교통 안내,B2033,고객,3,버스노선,,A,,,가산동 주민센터입니다.,,"가산동, 주민센터",가산동/동네/ 주민센터/공공기관,"주민센터,공공기관"
3,다산콜센터,대중교통 안내,B2033,상담사,4,,버스노선,A,,,,가산동 주민센터에서 남대문시장으로 가는 버스노선은 505번 버스입니다.,"가산동, 주민센터, 남대문시장,버스, 노선",가산동/동네/ 주민센터/공공기관/ 남대문시장/지명/ 버스/교통수단,"주민센터,교통수단"
4,다산콜센터,대중교통 안내,B2033,고객,5,버스정류장,,Q,어느정류장에서 타야합니까?,,,,정류장,,정류장


In [49]:
# 고객질문(요청) 컬럼의 데이터들의 개수를 확인
total_df['고객질문(요청)'].value_counts()
# 공백의 데이터가 여러개

고객질문(요청)
                               58725
카드결제와 현금결제할 때 요금 차이가 있나요?         79
                                  78
버스요금은 얼마인가요?                      55
시간은 얼마나 걸려요?                      54
                               ...  
중앙대학교에서 대법원으로 가는 버스를 알고싶습니다        1
법원까지 얼마나 걸립니까?                     1
비용은 얼마입니까?                         1
독산동에서 인사동으로 가는 버스를 알고싶습니다.         1
1순위는 누가 되나요?                       1
Name: count, Length: 22824, dtype: int64

In [50]:
total_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89302 entries, 0 to 89301
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   도메인        89302 non-null  object
 1   카테고리       89302 non-null  object
 2   대화셋일련번호    89302 non-null  object
 3   화자         89302 non-null  object
 4   문장번호       89302 non-null  int64 
 5   고객의도       89302 non-null  object
 6   상담사의도      89302 non-null  object
 7   QA         89302 non-null  object
 8   고객질문(요청)   89302 non-null  object
 9   상담사질문(요청)  89302 non-null  object
 10  고객답변       89302 non-null  object
 11  상담사답변      89302 non-null  object
 12  개체명        89302 non-null  object
 13  용어사전       89302 non-null  object
 14  지식베이스      89302 non-null  object
dtypes: int64(1), object(14)
memory usage: 10.2+ MB


In [51]:
# 공백으로 이루어진 value들을 통일화
# 스리즈에서 각각의 value을 추출하여 함수에 대입 -> map()
total_df = total_df.map(
    lambda x : str(x).strip()
)

In [52]:
# 1번 조건식 -> 현재 행에서 고객질문(요청) 데이터가 ''가 아니고 다음생의 상담사답변의 value가 ''이 아닌경우
flag1 = (total_df['고객질문(요청)'] != '') & (total_df['상담사답변'].shift(-1) != '')

In [53]:
# 2번 조건식 -> 현재 행에서 상담사답변이 ''가 아니고 전 행의 고객질문(요청) 데이터가 ''가 아닌 경우
flag2 = (total_df['상담사답변'] != '') & (total_df['고객질문(요청)'].shift(1) != '')

In [54]:
# flag1은 고객 질문만 나오고 flag2는 상담사 답변
# 둘 중 하나만 True라면
total_df = total_df.loc[flag1 | flag2, ]

In [55]:
# 고객 질문 데이터 중 중복 데이터는 제거
total_df['상담사답변'] = total_df['상담사답변'].shift(-1)

In [56]:
# 고객질문(요청) 데이터에서 ''가 아닌 데이터만 필터
total_df = total_df.loc[
    total_df['고객질문(요청)'] != ''
]

In [57]:
total_df = total_df.drop_duplicates('고객질문(요청)').reset_index(drop=True)

In [58]:
total_df

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,대중교통 안내,B2033,고객,5,버스정류장,,Q,어느정류장에서 타야합니까?,,,문성초등학교 정류장에서 탑승하시면 됩니다.,정류장,,정류장
1,다산콜센터,대중교통 안내,B2033,고객,7,버스요금,,Q,버스요금은 얼마입니까?,,,1200원 입니다.,"버스, 요금",버스/교통수단/ 요금/돈,"요금,돈"
2,다산콜센터,대중교통 안내,B2034,고객,1,버스노선,,Q,서울역에서 서울대학교가는 버스노선을 알고싶습니다.,,,"서울역에서 서울대학교로 가는 버스노선은 750A, 750B 버스입니다.","서울, 역, 서울대학교, 버스, 노선",서울/지명/ 역/역사/ 서울대학교/지명/ 버스/교통수단,"역,교통수단"
3,다산콜센터,대중교통 안내,B2034,고객,3,버스시간,,Q,시간은 얼마정도 걸립니까?,,,약 1시간 10분정도 걸립니다.,시간,,시간
4,다산콜센터,대중교통 안내,B2034,고객,7,버스요금,,Q,버스 요금은 얼마입니까?,,,교통카드로 1200원 입니다.,"버스, 요금",버스/교통수단/ 요금/돈,"요금,돈"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18947,다산콜센터,일반행정 문의,B35809,고객,9,여성전용아파트,,Q,꼭 서울시에 근무해야하나요?,,,서울소재 직장근무로 제한하고있습니다.,"서울시, 근무","서울시/서울/서울특별시, 근무",근무
18948,다산콜센터,일반행정 문의,B35809,고객,11,여성전용아파트,,Q,임대료는 어떻게 되나요?,,,62400원 입니다.,임대료,"임대료, 월세","임대료,월세"
18949,다산콜센터,일반행정 문의,B35809,고객,13,여성전용아파트,,Q,보증금도 있나요?,,,1423200원 입니다.,보증금,보증금/예치금,"보증금,예치금"
18950,다산콜센터,일반행정 문의,B35809,고객,17,여성전용아파트,,Q,입주순위도 있나요?,,,1~3순위가 있습니다.,"입주, 순위","입주/이사, 순위/순서","순위,순서"


In [59]:
# 토큰화, 벡터화 정의
from konlpy.tag import Komoran
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [60]:
komoran = Komoran()

def tokenize(text):
    return komoran.morphs(text)

vectorizer = TfidfVectorizer(
    tokenizer=tokenize,
    lowercase=False,
    ngram_range=(1,1),
    min_df=5,
    max_df=0.8
)

In [61]:
# total_df에서 고객질문(요청)데이터를 토큰화, 벡터화 작업
X = vectorizer.fit_transform(
    total_df['고객질문(요청)'].values
)

c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [62]:
# 질문 목록
new_questions = [
    '여권 재발급 신청 방법을 알려줘',
    '전입 신고가 인터넷으로 가능한가요?',
    '지방세 환급금을 어디서 신청하나요?'
]

In [63]:
# new_questions도 토큰, 벡터화 -> fit() x
test = vectorizer.transform(new_questions)

In [64]:
# 코사인 유사도 생성
sims = cosine_similarity(test, X)

In [65]:
sims

array([[0.        , 0.        , 0.03641695, ..., 0.        , 0.        ,
        0.        ],
       [0.02427217, 0.        , 0.03673714, ..., 0.        , 0.        ,
        0.05143491],
       [0.02110636, 0.        , 0.03996141, ..., 0.03383102, 0.03217321,
        0.02175454]], shape=(3, 18952))

In [66]:
for idx, sim in enumerate(sims):
    # 질문
    question = new_questions[idx]
    print('유저의 질문 :', question)
    # sim 데이터에서 내림차순정렬을 한 인덱스의 목록
    sim_idxs = sim.argsort()[::-1]
    for i in sim_idxs[:2]:
        # i :유저의 질문에 가장 유사한 질문의 인덱스
        print(f'유사도 : {round(sim[i], 3)},  유사 질문 : {total_df.loc[i, '고객질문(요청)']}, 답변 : {total_df.loc[i, '상담사답변']}')

유저의 질문 : 여권 재발급 신청 방법을 알려줘
유사도 : 0.619,  유사 질문 : 신청방법을 알려주세요., 답변 : 주민등록상 세대주와 가까운 주민센터 또는 복지로 홈페이지에서 신청가능하세요.
유사도 : 0.568,  유사 질문 : 신청방법 좀 알려주세요?, 답변 : 우선 사이트에 접속하셔서 회원가입을 해주세요. 청소년일경우 공인인증서가 없으면 본인확인절차를 거쳐 회원가입을 하고, 부모님이나 세대주분께서 가입을 하실 경우 공인인증서로 가입할 수 있습니다.
유저의 질문 : 전입 신고가 인터넷으로 가능한가요?
유사도 : 0.738,  유사 질문 : 인터넷으로도 신고가능한가요?, 답변 : 방문 접수밖에 안됩니다.
유사도 : 0.682,  유사 질문 : 인터넷으로 가능한가요?, 답변 : 인터넷으로 신청 가능합니다.
유저의 질문 : 지방세 환급금을 어디서 신청하나요?
유사도 : 0.76,  유사 질문 : 지방세 환급금 신청은 어떻게 해야하죠?, 답변 : 인터넷에서 접수를 하셔야 합니다
유사도 : 0.566,  유사 질문 : 환급금을 기부할 수도 있나요?, 답변 : 네 환급금을 사회복지공동모금회에 본인 명의로 기부가 가능합니다


### 문제 2
- 고객질문의 데이터를 이용하여 카테고리를 분류하는 모델을 생성
    - 고객질문 데이터들을 이용하여 토큰화, 벡터화 작업(독립변수)
    - 카테고리 일반행정, 대중교통을 타켓 데이터(종속변수)
        - 카테고리 데이터를 LabelEncoder()를 이용하여 수치화 변환
    - SVC 모델을 이용하여 벡터화된 데이터와 카테고리 데이터를 이용하여 학습
    - new_questions의 카테고리를 확인

In [67]:
# 독립변수
X = total_df['고객질문(요청)']
# 종속변수
Y = total_df['카테고리']

In [68]:
# 독립 변수를 토큰화, 벡터화 작업 -> SVC 모델에 학습
X_vec = vectorizer.fit_transform(X)
# 종속변수 -> LabelEncoder()를 이용하여 문자형 데이터를 숫자형 데이터로 변한
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
Y_le = le.fit_transform(Y)

In [69]:
X_vec.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(18952, 2111))

In [70]:
from sklearn.svm import SVC
# SVC 모델 객체를 생성
svc = SVC(
    random_state=42
)

In [71]:
svc.fit(X_vec, Y_le)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [72]:
# 질문 목록에 있는 데이터들을 토큰화, 벡터화 작업을 하고 학습이 된 svc 모델을 이용하여 예측 -> (0, 1) -> LabelEncoder를 이용하여 원본의 데이터로 다시 변환
new_vec = vectorizer.transform(new_questions)
pred = svc.predict(new_vec)

In [73]:
le.inverse_transform(pred)

array(['일반행정 문의', '일반행정 문의', '일반행정 문의'], dtype=object)

In [74]:
# 질문 목록
new_questions = [
    '여권 재발급 신청 방법을 알려줘',
    '전입 신고가 인터넷으로 가능한가요?',
    '지방세 환급금을 어디서 신청하나요?',
    '서울역에서 영등포로 가려면 어떻게 가나요?'
]

- new_questions 데이터를 이용하여 svc모델로 예측
- 예측 값을 이용하여 total_df의 카테고리 필터링
- 고객질문 모음을 토큰화, 벡터화 작업
- new_questions의 유사한 질문과 답변을 상위 2개만 출력

In [75]:
# 토큰화, 벡터화
test_vec = vectorizer.transform(new_questions)
# 모델을 이용하여 예측 생성
pred2 = svc.predict(test_vec)
pred2_origin = le.inverse_transform(pred2)
pred2_origin

array(['일반행정 문의', '일반행정 문의', '일반행정 문의', '대중교통 안내'], dtype=object)

In [85]:
# test_vec -> TD-IDF 방식으로 벡터화한 데이터셋 길이 (4)
# pred2_origin -> 예측 값들의 원본 데이터 길이 (4)
# 두 개의 변수를 이용하여 반복문 생성
# for idx in range(4):
#     print(test_vec[idx])
#     print(pred2_origin[idx])
#     break
for vec_data, cate in zip(test_vec, pred2_origin):
    # print(vec_data)
    # print(cate)
    # break
    # cate를 기준으로 total_df에서 카테고리 필터링 -> 벡터화 작업
    X_train = vectorizer.transform(
        total_df.loc[total_df['카테고리'] == cate, '고객질문(요청)']
    )

    # 코사인 유사도 (질문이 하나) -> (2차원 데이터 1차원으로 변경)
    sims = cosine_similarity(vec_data, X_train).ravel()
    # 유사도를 내림차순 정렬의 형태로 인덱스의 값들을 확인
    idxs = sims.argsort()[::-1]
    # 유사도 리스트에서 유사도가 높은 상위 2개만 출력하여 유사 질문 답변을 출력
    for idx in idxs[:2]:
        # print(idx)
        # idx는 array의 위치 값
        # 카테고리별로 필터링 된 데이터프레임에서 array와 같이 인덱스는 위치로 변환
        # iloc를 이용하여 유사 질문을 출력
        print('유사도 :', round(sims[idx], 3))
        print("유사 질문 :", total_df.loc[total_df['카테고리'] == cate, '고객질문(요청)'].iloc[idx])
        print("유사 답변 :", total_df.loc[total_df['카테고리'] == cate, '상담사답변'].iloc[idx])

유사도 : 0.619
유사 질문 : 신청방법을 알려주세요.
유사 답변 : 주민등록상 세대주와 가까운 주민센터 또는 복지로 홈페이지에서 신청가능하세요.
유사도 : 0.568
유사 질문 : 신청방법 좀 알려주세요?
유사 답변 : 우선 사이트에 접속하셔서 회원가입을 해주세요. 청소년일경우 공인인증서가 없으면 본인확인절차를 거쳐 회원가입을 하고, 부모님이나 세대주분께서 가입을 하실 경우 공인인증서로 가입할 수 있습니다.
유사도 : 0.738
유사 질문 : 인터넷으로도 신고가능한가요?
유사 답변 : 방문 접수밖에 안됩니다.
유사도 : 0.682
유사 질문 : 인터넷으로 가능한가요?
유사 답변 : 인터넷으로 신청 가능합니다.
유사도 : 0.76
유사 질문 : 지방세 환급금 신청은 어떻게 해야하죠?
유사 답변 : 인터넷에서 접수를 하셔야 합니다
유사도 : 0.566
유사 질문 : 환급금을 기부할 수도 있나요?
유사 답변 : 네 환급금을 사회복지공동모금회에 본인 명의로 기부가 가능합니다
유사도 : 0.671
유사 질문 : 서울역에서 발산역 지하철로어떨게 가나요?
유사 답변 : 서울역에서 공항철도지하철을 이용하셔서 김포공항에 내리시고 5호선환승하셔서 발산역에서 내리시면됩니다
유사도 : 0.571
유사 질문 : 어떻게 가나요?
유사 답변 : 오류역에서 지하철을 탄 후 대전역 지하철에서 내리신 후 14번 버스를 타면됩니다.


In [77]:
total_df.loc[total_df['카테고리'] == '일반행정 문의', '고객질문(요청)']

6688     지방세를 내려면 어떻게 해야됩니까?
6689         은행을 직접방문해도 됩니까?
6690        버스로 가는 방법도 있습니까?
6691          다른 납부방법도 있습니까?
6692        사이트 주소가 어떻게 됩니까?
                ...         
18947        꼭 서울시에 근무해야하나요?
18948          임대료는 어떻게 되나요?
18949              보증금도 있나요?
18950             입주순위도 있나요?
18951           1순위는 누가 되나요?
Name: 고객질문(요청), Length: 12264, dtype: object